# 16 SMDPL: the TNG-trained models on another simulation

The real-space model (the published one) and the two redshift-space models (line of sight z only, and z + y) predict the whole SMDPL box from its Rockstar halos, and are scored against T-web labels computed from SMDPL's own dark-matter density. Nothing from SMDPL was used in training, so the whole box is a test set, and in redshift space every line of sight (x, y and z) is a fair test.

**The grid problem.** CosmoSim gives the SMDPL density only on 512³ cells of 0.78 h⁻¹ Mpc, twice the 0.40 h⁻¹ Mpc the models were trained on. So:

- the halos are gridded on 1024³ cells (0.39 h⁻¹ Mpc), which the models accept;
- the labels are computed on SMDPL's 512³ grid, with the same rule as for TNG (effective smoothing 1 h⁻¹ Mpc, threshold 0.65);
- the predicted class scores are averaged onto SMDPL's 512³ grid points (weights 1/4, 1/2, 1/4 along each axis, centered on each point) and scored there.

The same procedure is checked on TNG, where the right answer is known: each model's TNG test-slab prediction is averaged onto a 256³ grid (0.80 h⁻¹ Mpc) and scored against the TNG 256³ mask (computed the same way from the 256³ dark matter). The TNG 256³ scores, not the usual 512³ ones, are the fair reference for SMDPL.

**Where SMDPL's grid points sit** is not documented: point *i* could be at *i* × 0.78 h⁻¹ Mpc or at the middle of cell *i*. Half a cell is one cell of the model's grid, enough to cost real score. Section 3 measures it from the data.

**Runtime.** **L4 GPU, High-RAM.** Each 1024³ prediction takes several minutes; all of them (4 spacings × (1 real + 2 models × 3 lines of sight)) take a few hours. Results are saved after each prediction and finished ones are skipped, so the notebook can be rerun after a disconnect. Set `SPACINGS`, `LINES_OF_SIGHT` and `MODELS` below to run a subset.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!rm -rf /content/deepvoid
!git clone -q "https://github.com/skumagai3/deepvoid.git" /content/deepvoid
!pip -q install --force-reinstall --no-deps /content/deepvoid
!git -C /content/deepvoid log --oneline -1

In [ ]:
import gc
import json
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from deepvoid import Catalog
from deepvoid.evaluate import confusion_matrix, scores
from deepvoid.grid import cic
from deepvoid.io import load_volume, save_volume
from deepvoid.labels import CLASS_NAMES, tweb_labels
from deepvoid.observe import los_to_z, select_by_spacing, to_redshift_space
from deepvoid.pretrained import Pretrained
from deepvoid.resample import block_mean, onto_coarse_points
from deepvoid.sources.smdpl import BOX_SIZE, read_halos
from deepvoid.split import Region
from deepvoid.volume import Volume

DATA = Path('/content/drive/MyDrive/data/SMDPL')
HALO_FILE = DATA / '2026-11-15-22-18-55-728229.parquet'   # the CosmoSim halo download
DENSITY_FILE = DATA / 'smdpl_dens512_z0.npy'                # notebook 15
LABEL_FILE = DATA / 'smdpl_mask_tweb_th0.65_seff1.npz'      # written below once
RUNS = Path('/content/drive/MyDrive/deepvoid_runs')
OUT = RUNS / 'smdpl'
OUT.mkdir(exist_ok=True)
RESULTS = OUT / 'results.json'

SPACINGS = [3, 5, 7, 10]
LINES_OF_SIGHT = {'x': 0, 'y': 1, 'z': 2}
N_FINE = 1024                       # model grid: 400 / 1024 = 0.39 h^-1 Mpc
KEYS = ['mcc', 'void_mcc', 'void_f1', 'wall_f1', 'filament_f1', 'halo_f1']
catalog = Catalog()

rsd_z = [f'tng300-3/subhalos/L{s}/rsd-z' for s in SPACINGS]
MODELS = {   # label: (how to open it, real or redshift space)
    'real space': (lambda: Pretrained.from_folder(RUNS.parent / 'deepvoid_model_release'), 'real'),
    'redshift space, z': (lambda: Pretrained.from_run(RUNS / 'subhalo_C_mixed_smooth2ch_depth5_rsd',
                                                      catalog, inputs=rsd_z), 'rsd'),
    'redshift space, y+z': (lambda: Pretrained.from_run(RUNS / 'subhalo_C_mixed_smooth2ch_depth5_rsd_yz',
                                                        catalog, inputs=rsd_z), 'rsd'),
}
results = json.loads(RESULTS.read_text()) if RESULTS.exists() else {}
print(len(results), 'results already saved')


def save_result(key, cm):
    results[key] = {**{k: scores(cm)[k] for k in KEYS}, 'confusion_matrix': cm.tolist()}
    RESULTS.write_text(json.dumps(results, indent=1))
    print(f'{key:45s}', {k: round(v, 3) for k, v in results[key].items() if k in KEYS}, flush=True)

## 1. Halos

All Rockstar halos and subhalos, as TNG's Subfind subhalos include both. λ = 3 needs (400/3)³ ≈ 2.37 million of them.

In [ ]:
halos = read_halos(HALO_FILE)
need = round((BOX_SIZE / min(SPACINGS)) ** 3)
print(f'{len(halos):,} halos ({halos.meta.get("n_subhalos", "?"):,} subhalos); λ = {min(SPACINGS)} needs {need:,}')
print(f'mvir {halos.mass.min():.3g} to {halos.mass.max():.3g} h^-1 Msun; '
      f'positions {halos.pos.min():.2f} to {halos.pos.max():.2f} h^-1 Mpc')
print('velocity rms along x, y, z:', ', '.join(f'{halos.vel[:, a].std():.0f}' for a in range(3)), 'km/s')
assert len(halos) >= need

## 2. Density and labels

`dens` is ρ/ρ̄ (mean 1, checked in notebook 15). The labels use the same rule as TNG's: effective smoothing 1 h⁻¹ Mpc (here a Gaussian of 0.80 cells, since the cell itself counts for 0.78 h⁻¹ Mpc), threshold 0.65 on the tidal eigenvalues. Computed once and saved. The class fractions are compared with TNG's at 512³ and at 256³ (the closer grid).

In [ ]:
dens = np.load(DENSITY_FILE)
print('density', dens.shape, dens.dtype, f'mean {dens.mean():.4f}, min {dens.min():.4f}')
if LABEL_FILE.exists():
    labels = load_volume(LABEL_FILE, dtype=None).data
else:
    labels_vol = tweb_labels(Volume(data=dens, box_size=BOX_SIZE, meta={'sim': 'smdpl'}),
                             sigma_eff=1.0, threshold=0.65)
    save_volume(labels_vol, LABEL_FILE)
    labels = labels_vol.data
    print('sigma', round(labels_vol.meta['sigma_cells'], 3), 'cells; saved', LABEL_FILE)

def fractions(a):
    return np.bincount(a.ravel(), minlength=4) / a.size

frac = pd.DataFrame({'SMDPL 512^3 (0.78)': fractions(labels),
                     'TNG 256^3 (0.80)': fractions(catalog.load('tng300-3/n256/mask').data),
                     'TNG 512^3 (0.40)': fractions(catalog.load('tng300-3/mask').data)},
                    index=CLASS_NAMES)
print(frac.round(3).to_string())

## 3. Where SMDPL's grid points are

The halos (λ = 3, real space) are gridded on 1024³ cells with our convention (point *i* at *i* × 0.39 h⁻¹ Mpc), averaged onto a 512³ grid with its points either on even fine points (offset 0: point *i* at *i* × 0.78) or on odd ones (offset 1: the middle of cell *i*), and correlated with the SMDPL density, both as log(1 + ρ/ρ̄). The axis order is checked too: the density should match best with its axes as stored. The best combination should stand out; if two are close, stop and look before going on.

In [ ]:
import itertools
sample = select_by_spacing(halos, min(SPACINGS))
fine = cic(sample, N_FINE).data
fine /= fine.mean()
logd = np.log1p(dens).ravel()
logd = (logd - logd.mean()) / logd.std()
rows = []
for offset in (0, 1):
    h = np.log1p(onto_coarse_points(fine, offset))
    for perm in itertools.permutations(range(3)):
        v = np.transpose(h, perm).ravel()
        r = float(np.mean((v - v.mean()) / v.std() * logd))
        rows.append({'offset': offset, 'axes': ''.join('xyz'[p] for p in perm), 'corr': r})
    del h
align = pd.DataFrame(rows).sort_values('corr', ascending=False)
print(align.to_string(index=False))
best = align.iloc[0]
assert best['axes'] == 'xyz', 'the density matches best with its axes reordered; look before going on'
OFFSET = int(best['offset'])
gap = best['corr'] - align[(align['axes'] == 'xyz') & (align['offset'] != OFFSET)]['corr'].iloc[0]
print(f'using offset {OFFSET} (correlation better by {gap:.4f} than the other offset)')
del fine, sample
gc.collect()

## 4. TNG reference on a 0.80 h⁻¹ Mpc grid

Each model on its usual TNG test-slab input (real-space or z-line-of-sight). Scored twice: on the 512³ test slab as usual (this should reproduce each run's known scores), and with the class scores averaged in 2×2×2 blocks onto the 256³ grid against the TNG 256³ mask, the same averaging used to make that mask's density.

In [ ]:
split = json.loads((RUNS / 'subhalo_C_mixed_smooth2ch_depth5' / 'split.json').read_text())
test512 = Region(**split['test'])
test256 = Region(test512.name, test512.axis, test512.start // 2, test512.stop // 2)
assert test512.start % 2 == 0 and test512.stop % 2 == 0
mask512 = catalog.load('tng300-3/mask').data
mask256 = catalog.load('tng300-3/n256/mask').data

def slab(a, region):
    sl = [slice(None)] * 3
    sl[region.axis] = slice(region.start, region.stop)
    return a[tuple(sl)]

for label, (open_model, space) in MODELS.items():
    if all(f'TNG|{label}|L{s}|{g}' in results for s in SPACINGS for g in ('512', '256')):
        continue
    model = open_model()
    for s in SPACINGS:
        if f'TNG|{label}|L{s}|256' in results:
            continue
        name = f'tng300-3/subhalos/L{s}' + ('' if space == 'real' else '/rsd-z')
        vol = catalog.load(name)
        probs = model.predict(vol.data, vol.box_size, float(s), out_dtype=np.float16)
        save_result(f'TNG|{label}|L{s}|512',
                    confusion_matrix(slab(mask512, test512).ravel(), slab(probs.argmax(-1), test512).ravel()))
        coarse = np.stack([block_mean(probs[..., c].astype(np.float32), 2) for c in range(4)], axis=-1)
        save_result(f'TNG|{label}|L{s}|256',
                    confusion_matrix(slab(mask256, test256).ravel(), slab(coarse.argmax(-1), test256).ravel()))
        del vol, probs, coarse
        gc.collect()
    del model

## 5. SMDPL

For each spacing: the most massive halos (as for TNG), gridded on 1024³ in real space and in redshift space along x, y and z. A line of sight along x or y is turned onto z for the redshift-space models (input swapped, prediction swapped back), as in training. Each prediction is averaged onto the SMDPL grid points (offset from section 3) and scored against the SMDPL labels over the whole box.

In [ ]:
assert 'OFFSET' in globals(), 'run section 3 first'
opened = {}
def get_model(label):
    if label not in opened:
        opened.clear(); gc.collect()
        opened[label] = MODELS[label][0]()
    return opened[label]

for s in SPACINGS:
    variants = ['real'] + list(LINES_OF_SIGHT)
    todo = [(v, label) for v in variants for label, (_, space) in MODELS.items()
            if (space == 'real') == (v == 'real') and f'SMDPL|{label}|L{s}|{v}' not in results]
    if not todo:
        continue
    sample = select_by_spacing(halos, s)
    for v in variants:
        labels_here = [label for (vv, label) in todo if vv == v]
        if not labels_here:
            continue
        t0 = time.time()
        if v == 'real':
            field = cic(sample, N_FINE).data
        else:
            axis = LINES_OF_SIGHT[v]
            field = np.ascontiguousarray(los_to_z(cic(to_redshift_space(sample, axis), N_FINE).data, axis))
        for label in labels_here:
            probs = get_model(label).predict(field, BOX_SIZE, float(s), out_dtype=np.float16)
            coarse = onto_coarse_points(probs, OFFSET)
            del probs
            if v != 'real':
                coarse = los_to_z(coarse, LINES_OF_SIGHT[v])     # back to the box's own axes
            save_result(f'SMDPL|{label}|L{s}|{v}', confusion_matrix(labels.ravel(), coarse.argmax(-1).ravel()))
            del coarse
            gc.collect()
        print(f'   λ = {s}, {v}: {time.time() - t0:.0f} s', flush=True)
        del field
        gc.collect()

## 6. Results

For each model and spacing: TNG test slab at 512³ (the usual score), TNG test slab at 256³ (the fair reference), and SMDPL (real space; or each line of sight and their mean in redshift space).

In [ ]:
def table(metric='mcc'):
    rows = {}
    for label, (_, space) in MODELS.items():
        for s in SPACINGS:
            r = {'TNG 512': results.get(f'TNG|{label}|L{s}|512', {}).get(metric),
                 'TNG 256': results.get(f'TNG|{label}|L{s}|256', {}).get(metric)}
            if space == 'real':
                r['SMDPL'] = results.get(f'SMDPL|{label}|L{s}|real', {}).get(metric)
            else:
                per = {v: results.get(f'SMDPL|{label}|L{s}|{v}', {}).get(metric) for v in LINES_OF_SIGHT}
                r.update({f'SMDPL {v}': x for v, x in per.items()})
                got = [x for x in per.values() if x is not None]
                r['SMDPL'] = float(np.mean(got)) if got else None
            rows[(label, s)] = r
    return pd.DataFrame(rows).T

for metric in ('mcc', 'void_mcc', 'halo_f1'):
    print(f'\n{metric}')
    print(table(metric).round(3).to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.3))
colors = {'real space': 'k', 'redshift space, z': '#dd6b20', 'redshift space, y+z': '#2b6cb0'}
for ax, metric, name in zip(axes, ('mcc', 'void_mcc'), ('four-class MCC', 'void MCC')):
    t = table(metric)
    for label in MODELS:
        sub = t.loc[label]
        ax.plot(SPACINGS, sub['TNG 256'], 'o--', color=colors[label], alpha=0.6, label=f'{label}: TNG, 0.80 grid')
        ax.plot(SPACINGS, sub['SMDPL'], 's-', color=colors[label], label=f'{label}: SMDPL, 0.78 grid')
    ax.set_xticks(SPACINGS); ax.set_xlabel(r'mean tracer separation $\lambda$ ($h^{-1}$ Mpc)')
    ax.set_title(name); ax.grid(alpha=0.3)
axes[0].legend(fontsize=7)
fig.suptitle('TNG-trained models on SMDPL, both scored on ~0.8 h⁻¹ Mpc grids')
fig.tight_layout()
fig.savefig(OUT / 'smdpl_scores_vs_spacing.png', dpi=150, bbox_inches='tight')
plt.show()
print('saved in', OUT)

## 7 Line-of-sight velocities: SMDPL halos against TNG subhalos

On SMDPL the real-space model scores about 0.02 higher than on TNG and the redshift-space models about 0.02 lower, so the difference is in the velocities, not the positions. This compares the line-of-sight shift $v_\parallel / 100$ (h⁻¹ Mpc at z = 0) of the tracers each model sees, at each spacing: its spread, its tail (fingers of God) and the share of satellites (Rockstar subhalos, `upid ≠ -1`; TNG subhalos that are not the first in their group). All three axes are pooled.

CPU only and independent of the sections above: on a fresh runtime, run the first cell (Drive and the package) and then this section.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from deepvoid import Catalog

catalog = Catalog()
SMDPL_HALOS = Path('/content/drive/MyDrive/data/SMDPL/2026-11-15-22-18-55-728229.parquet')
TNG_SUBHALOS = catalog.root / 'TNG' / 'TNG300-1_subhalos_snap99.npz'   # notebook 10
OUT = Path('/content/drive/MyDrive/deepvoid_runs/smdpl')
SPACINGS = [3, 5, 7, 10]

df = pd.read_parquet(SMDPL_HALOS)
df.columns = [c.lower() for c in df.columns]
smdpl = {'box': 400.0, 'mass': df['mvir'].to_numpy(np.float64),
         'vel': df[['vx', 'vy', 'vz']].to_numpy(np.float64),
         'satellite': df['upid'].to_numpy() != -1}
del df

cat = np.load(TNG_SUBHALOS)
print('TNG catalog fields:', sorted(cat.files))
tng = {'box': 205.0, 'mass': np.asarray(cat['SubhaloMass'], np.float64),
       'vel': np.asarray(cat['SubhaloVel'], np.float64), 'satellite': None}
if 'SubhaloGrNr' in cat.files and 'GroupFirstSub' in cat.files:
    first = np.asarray(cat['GroupFirstSub'])[np.asarray(cat['SubhaloGrNr'])]
    tng['satellite'] = first != np.arange(len(first))
else:
    print('no group fields in the TNG file: satellite share left blank')


def shift_stats(sim, spacing):
    # Same selection as select_by_spacing: the (box / spacing)^3 most massive.
    n = round((sim['box'] / spacing) ** 3)
    keep = np.argsort(sim['mass'])[-n:]
    shift = np.abs(sim['vel'][keep] / 100.0).ravel()   # h^-1 Mpc, three axes pooled
    row = {'tracers': n, 'rms': float(np.sqrt(np.mean(shift ** 2))),
           'median': float(np.median(shift)), 'p90': float(np.percentile(shift, 90)),
           'p99': float(np.percentile(shift, 99)), '> 5': float(np.mean(shift > 5)),
           '> 10': float(np.mean(shift > 10))}
    if sim['satellite'] is not None:
        row['satellites'] = float(sim['satellite'][keep].mean())
    return row, shift


rows, shifts = {}, {}
for s in SPACINGS:
    for name, sim in (('TNG', tng), ('SMDPL', smdpl)):
        rows[(s, name)], shifts[(s, name)] = shift_stats(sim, s)
vel_table = pd.DataFrame(rows).T
vel_table.index.names = ['λ', 'sim']
print('line-of-sight shift |v| / 100, h^-1 Mpc (> 5, > 10: share of tracer-axes beyond that)')
print(vel_table.round(3).to_string())
(OUT / 'velocities.json').write_text(json.dumps({f'L{s}|{n}': r for (s, n), r in rows.items()}, indent=1))

In [ ]:
fig, axes = plt.subplots(1, len(SPACINGS), figsize=(4 * len(SPACINGS), 3.4), sharey=True)
bins = np.linspace(0, 20, 81)
for ax, s in zip(axes, SPACINGS):
    for name, color in (('TNG', 'k'), ('SMDPL', '#dd6b20')):
        ax.hist(shifts[(s, name)], bins=bins, density=True, histtype='step', color=color, label=name)
    ax.set_yscale('log'); ax.set_title(rf'$\lambda$ = {s}'); ax.set_xlabel(r'$|v_\parallel| / 100$ ($h^{-1}$ Mpc)')
axes[0].set_ylabel('density'); axes[0].legend()
fig.tight_layout()
fig.savefig(OUT / 'velocities.png', dpi=150, bbox_inches='tight')
plt.show()

**Reading it.** If SMDPL's shifts are larger (higher rms, longer tail, more satellites), its tracers are displaced more than anything the redshift-space models saw in training, which would explain the drop. If the two match, the cause is elsewhere (for example how Rockstar and SUBFIND assign a halo's velocity) and the drop needs another look. The larger SMDPL box also carries more large-scale flow, which can raise the rms on its own.

## Shut down

Deletes this runtime once every cell above has finished, so it stops using compute units. The results are already on Drive; anything kept only under `/content` goes with the runtime. With **Run all**, Colab stops at the first cell that fails, so after an error the runtime stays up and you can look at it. Set `SHUT_DOWN = False` to keep the runtime.

In [ ]:
SHUT_DOWN = True   # False keeps the runtime after the run
if SHUT_DOWN:
    import time
    from google.colab import drive, runtime
    drive.flush_and_unmount()   # finish writing to Drive first
    time.sleep(5)
    runtime.unassign()          # disconnects and deletes this runtime